[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02/Quest.ipynb)


<!-- gen:values.yml#weeks.02.notebook-title|Quest 02: The Coffee Cart Ledger -->
# Week 2 Quest: The Coffee Cart Ledger
<!-- /gen -->

*Your first real client engagement: A week of till records from Beans of Egham, and a founder who needs numbers she can trust.*

**About this [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest).** It takes about 90 minutes. Everything you need is in this week's [Demonstration](./Demonstration.ipynb), especially dictionaries, the counting pattern, and grouping with `elif`. Work with your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) (GitHub Copilot's chat panel) throughout. This quest practises **[GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge)** and **[PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run)**, the two moves introduced this week, plus one round each of [BUG-HUNT](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bug-hunt) and [TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back). Keep your best and worst exchanges for the [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) cell at the end.

## 📋 Stage 1: Briefing

*(about 10 minutes)*

> **From:** A. Partner, [Runnymede Analytics](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics)
> **To:** You ([Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst))
> **Re:** Beans of Egham: The coffee cart ledger
>
> The founder of *Beans of Egham* (the campus coffee cart) has sent us her first full week of till records. There are 150 transactions, exported from the card reader as a list of records. She is deciding **what to stock more of** and **whether her trial late opening is worth keeping**. She meets her supplier on Friday.
>
> She wants three things: Her **total revenue** for the week, her **best-selling drink**, and **revenue broken down by time of day**. One warning: Her regulars buy *rounds*, so one transaction is often several cups. Do not confuse transactions with cups. Verify before you ship.
>
> A.P.

**The business question:** What sold, when, and for how much, counted so that the founder can check it against her till roll?

**The deliverable:** Printed, penny-accurate figures for total revenue, best-selling drink (in **cups**, not transactions), and revenue by time-of-day slot.

**[Definition of Done](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-definition-of-done)**: Tick these yourself before claiming a [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge):

- [ ] Every ✏️ prediction cell in [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) filled in *before* its code cell was run
- [ ] Total weekly revenue printed as `£x.xx`, computed as **cups × price per cup**, summed over all 150 transactions
- [ ] Best-selling drink identified **by cups sold**, with the cup count printed
- [ ] One hand-check passed: Your loop's total over `sales[:2]` matches a calculator
- [ ] Cross-total check passed: The values in your cups-per-item dictionary sum to your total cups, not to 150 (the number of transactions)
- [ ] The founder's summary written to `assets/data/week02/founder_summary.txt` and read back in to prove it saved
- [ ] The [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) bug found, explained **in business terms**, and fixed by directing the Associate
- [ ] All three [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief) prompts answered, and the AI Use Record cell filled in
- [ ] A dated entry in your `QuestLog.md` with the badge you are claiming

**Badge criteria this week:**

- 🥉 **[Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze)**: Total revenue and best-selling drink (by cups), built with your Associate, verified with all three checks, and saved to the founder's summary file
- 🥈 **[Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver)**: Bronze, *plus* judging the two Associate drafts for revenue-by-slot. You choose one, defend the choice in writing, and adapt the chosen draft to the founder's new **twilight** slot
- 🥇 **[Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold)**: Silver, *plus* a loyalty-discount simulation of your own design

*Stage timings: Briefing 10 · First look 15 · Build 40 · Red Flags 10 · Debrief 15 minutes. These are guidance, not a stopwatch.*

### The data

Run the cell below first. It generates the founder's week of till records with the module's fixed seed (`5813`). Every analyst and every marker sees the same 150 transactions. The comment block at the top is the **schema**. It names each field in a record and says what it means. Paste that comment block into your Associate prompts.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# Beans of Egham: One week of till records, as a list of dictionaries.
# Each record is ONE TRANSACTION (one customer order). Keys:
#   item        : The drink ordered
#   size        : "regular" or "large" (large costs £0.40 more per cup)
#   cups        : How many cups in the order (regulars buy rounds for the office)
#   price       : The price PER CUP in pounds, size included
#   time_of_day : When the sale happened, 24-hour clock, e.g. "08:35"
import os
import numpy as np

os.makedirs("assets/data/week02", exist_ok=True)  # where Bronze writes the founder's summary
rng = np.random.default_rng(5813)  # module seed: Do not change

menu = {
    "flat white": 3.40, "latte": 3.60, "cappuccino": 3.50,
    "espresso": 2.20, "mocha": 3.80, "hot chocolate": 3.20,
    "english breakfast tea": 2.40, "filter coffee": 2.80,
}
items = list(menu)
item_weights = [0.24, 0.17, 0.14, 0.08, 0.10, 0.07, 0.12, 0.08]
slot_hours = {"morning": [7, 8, 9, 10], "lunchtime": [11, 12, 13],
              "afternoon": [14, 15], "twilight": [16, 17, 18]}
slot_weights = [0.42, 0.30, 0.20, 0.08]
slot_names = list(slot_hours)

sales = []
for _ in range(150):
    item = items[int(rng.choice(len(items), p=item_weights))]
    size = "large" if rng.random() < 0.3 else "regular"
    if item == "latte":
        cups = int(rng.choice([1, 2, 3], p=[0.30, 0.45, 0.25]))
    else:
        cups = int(rng.choice([1, 2, 3], p=[0.72, 0.20, 0.08]))
    slot = slot_names[int(rng.choice(len(slot_names), p=slot_weights))]
    hour = int(rng.choice(slot_hours[slot]))
    minute = int(rng.integers(0, 60))
    price = menu[item] + (0.40 if size == "large" else 0.00)
    sales.append({
        "item": item,
        "size": size,
        "cups": cups,
        "price": round(price, 2),
        "time_of_day": f"{hour:02d}:{minute:02d}",
    })

print(f"Created sales: {len(sales)} till records from one week at Beans of Egham")
print(f"Example record: {sales[0]}")

## 🔍 Stage 2: First look

*(about 15 minutes)*

Your Associate took a first look overnight and left four cells. **PREDICT-RUN** each one: Write one line in the ✏️ cell *before* you run the code. Then run it and note any surprise. At least one of these cells shows you something you need in Red Flags.

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 1: How big is the week, and what does one record look like?
print(len(sales))
print(sales[0])

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 2: The first three records, as readable receipt lines.
for sale in sales[:3]:
    print(f"{sale['cups']} x {sale['size']} {sale['item']}"
          f" at £{sale['price']:.2f} each ({sale['time_of_day']})")

> ✏️ **Your prediction:** *(one line: What will this cell output? Eight lines? 150? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 3: How many TRANSACTIONS does each drink appear in?
transactions_per_item = {}
for sale in sales:
    item = sale["item"]
    transactions_per_item[item] = transactions_per_item.get(item, 0) + 1

for item, count in transactions_per_item.items():
    print(f"{item}: {count} transactions")

> ✏️ **Your prediction:** *(one line: Roughly what fraction of orders will be more than one cup? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 4: The Partner's warning, checked. How common are multi-cup rounds?
multi_cup = 0
for sale in sales:
    if sale["cups"] > 1:
        multi_cup = multi_cup + 1

print(f"{multi_cup} of {len(sales)} transactions are rounds of 2+ cups")

**First look findings.** Answer each of these in one sentence. How many records are there? What are a record's keys? What did First look 4 show you about the difference between *transactions* and *cups*? Carry that last one into everything below.

## 🛠️ Stage 3: Build

*(about 40 minutes)*

Aim for at least Bronze, with Silver being the expected standard.

Use your Associate throughout. Paste the schema comment from the data-setup cell into your first prompt.

### 🥉 Bronze

**Deliverable:** One cell that loops over `sales` and does three things:

- Prints **total weekly revenue** as `£x.xx`
- Prints the **best-selling drink by cups sold**, with its cup count
- Writes both figures to `assets/data/week02/founder_summary.txt`, then reads the file back to prove it saved (the Demonstration's file round-trip)

This is **GENERATE-JUDGE**. A suggested prompt to send your Associate:

> *"I have a Python list called `sales` where each element is a dict like `{'item': 'flat white', 'size': 'regular', 'cups': 2, 'price': 3.40, 'time_of_day': '08:35'}`. `price` is per cup, and `cups` can be more than 1. Using a beginner-level `for` loop and a dictionary (no libraries), compute the total revenue and the number of cups sold per item. Then print the total as £x.xx and the item with the most cups."*

Then **verify before you ship**:

1. **Hand-check:** Work out the first two records with a calculator (First look 2 printed them). In a spare cell, run your loop over `sales[:2]` and check it matches to the penny
2. **Plausibility check:** Divide your total revenue by the total number of *cups*. The answer must sit between £2.20 and £4.20, the cheapest and dearest prices on the menu. If it does not, your loop is counting something wrongly
3. **Cross-total check:** The values of your cups-per-item dictionary must sum to the same total-cups figure you used in the plausibility check. If they sum to 150, the number of *transactions*, you have counted the wrong thing, and your best seller is probably wrong too

In [ ]:
# 🥉 Bronze: Build it here with your Associate, then verify it.
# TODO: Total revenue = sum of cups * price over all records, printed as £x.xx
# TODO: Cups per item in a dictionary (the counting pattern), then the best seller
# TODO: Write both figures to assets/data/week02/founder_summary.txt,
#       then read the file back and print it (the Demonstration's round-trip)
# Remember: 'best-selling' is measured in CUPS, not transactions.

total_revenue = 0
cups_per_item = {}
# ... your loop here ...

pass

Keep the checks in their own cell, separate from the build, so you can rerun them after every change.

In [ ]:
# 🥉 Verification - hand-check on sales[:2] and the plausibility check.
# TODO: Run your revenue loop over just sales[:2]; compare with your calculator.
# TODO: Total revenue / total cups. Is it between £2.20 and £4.20?
# TODO: Sum the values of your cups-per-item dictionary. It must equal
#       your total cups, not 150 (150 means you counted transactions)

pass

### 🥈 Silver

The founder's second question: **"When do I actually make my money: Morning, lunchtime, or afternoon?"**

You asked your Associate twice and got two drafts back. Run both. They work, and they agree on the numbers. Your job is to **judge** which one you would keep, then adapt it.

The founder defines the slots as **morning** before 11:00, **lunchtime** 11:00–13:59, and **afternoon** from 14:00.

In [ ]:
# Associate draft A: Buckets with if/elif, running totals.
revenue_by_slot = {"morning": 0.0, "lunchtime": 0.0, "afternoon": 0.0}

for sale in sales:
    hour = int(sale["time_of_day"].split(":")[0])
    if hour < 11:
        slot = "morning"
    elif hour < 14:
        slot = "lunchtime"
    else:
        slot = "afternoon"
    revenue_by_slot[slot] = revenue_by_slot[slot] + sale["cups"] * sale["price"]

for slot, revenue in revenue_by_slot.items():
    print(f"{slot}: £{revenue:.2f}")

**Draft B** reaches the same totals by a different route. Run it before judging.

In [ ]:
# Associate draft B: A dict of lists. Keep every line, total at the end.
def slot_for(time_of_day):
    hour = int(time_of_day.split(":")[0])
    if hour < 11:
        return "morning"
    elif hour < 14:
        return "lunchtime"
    return "afternoon"

lines_by_slot = {"morning": [], "lunchtime": [], "afternoon": []}

for sale in sales:
    slot = slot_for(sale["time_of_day"])
    lines_by_slot[slot].append(sale["cups"] * sale["price"])

for slot, lines in lines_by_slot.items():
    average = sum(lines) / len(lines)
    print(f"{slot}: £{sum(lines):.2f} ({len(lines)} transactions, average £{average:.2f})")

**Your Silver task: Judge, defend, adapt.**

1. **Judge.** Ask your Associate to argue for each draft in turn. For example: *"Argue for A, then argue for B, then recommend one for a beginner analyst who'll extend this next month."* Then decide for yourself:
   - Which is easier to read?
   - Which gives you extra answers (transaction counts, averages) without extra work?
   - Which would you rather change in a hurry?
2. **Defend.** Write 2–3 sentences in the markdown cell below, in your own words, not the Associate's
3. **Adapt.** The founder is trialling late opening and wants a fourth slot the Associate was not told about: **Twilight, from 16:00 onwards** (afternoon becomes 14:00–15:59). Adapt *your chosen draft* in the cell below so all four slots print. Check first that sales after 16:00 exist, by counting records whose hour is 16 or more. If your twilight row then shows £0.00, it is a wrong answer, not an empty slot

**✍️ My choice and defence:** *(A or B, and why: 2–3 sentences in your own words)*

> 

In [ ]:
# 🥈 Silver - your chosen draft, adapted to add the twilight slot (16:00+).
# TODO: Paste your chosen draft here and modify it. Four slots should print.

pass

### 🥇 Gold

The founder, as you pack up: *"Everyone's telling me I need a loyalty scheme. Before I print stamp cards: What would one actually cost me?"*

**Deliverable:** A small **loyalty-discount simulation of your own design**, run against this week's `sales`. You choose the rule and direct the Associate to build it. Print the week's revenue **as-is** next to revenue **under your rule**, and state the rule in one sentence. Ideas you can use or ignore:

- Rounds of 2+ cups get 10% off
- Every drink in the twilight slot is half price
- 30p off every cup priced over £3.50
- A rule of your own

Gold has no upper limit. You could compare two rules, or find the rule that costs closest to 5% of revenue. It ends when you can print the two totals and defend the rule in a sentence.

In [ ]:
# 🥇 Gold - my loyalty rule: (state it here in one comment line)
# TODO: Design with your Associate; print revenue as-is vs revenue under the rule.

pass

## 🚩 Stage 4: Red Flags

*(about 10 minutes)*

**BUG-HUNT.** While you were building, the Associate drafted the founder's headline summary for the supplier meeting. It runs without an error and looks ready to send. Run it, then audit it against numbers you already trust from Bronze and First look.

In [ ]:
# The Associate's summary for the founder: It runs clean. Is it right?
total_revenue = 0
for sale in sales:
    total_revenue = total_revenue + sale["cups"] * sale["price"]

cups_sold = len(sales)
average_per_cup = total_revenue / cups_sold

print("=== Beans of Egham: the week in numbers ===")
print(f"Cups sold:        {cups_sold}")
print(f"Total revenue:    £{total_revenue:.2f}")
print(f"Average per cup:  £{average_per_cup:.2f}")

**Your audit, three steps:**

1. **Find it.** One number here contradicts what you proved in First look 4. The "average per cup" also fails Bronze's plausibility check: No drink on the menu costs that much. Which line is wrong?
2. **Explain it in business terms** in the cell below. Do not write 'it uses `len()`'. Write what the founder would get wrong on Friday. How many cups and how much milk would she order? What would she think a customer pays for a drink?
3. **Fix it by directing, not typing.** Tell your Associate what is wrong and why. For example: *"This summary counts transactions with `len(sales)` and calls them cups, but one transaction can be several cups. Fix it so cups are counted by summing the `cups` field, and explain the difference"*. Paste its corrected version into the fix cell

**✍️ The bug, in business terms:** *(2–3 sentences: What is wrong, and what would it cost the founder on Friday?)*

> 

In [ ]:
# 🚩 Fixed summary: Corrected by directing your Associate, not by hand-editing.
# TODO: Paste the corrected summary here and run it.

pass

## 📓 Stage 5: Debrief

*(about 15 minutes)*

**TEACH-BACK.** Answer all three in the cells below, then copy them into your `QuestLog.md` as a dated entry with the badge you are claiming.

**1. Client explanation:** "Explain this week's key technique to the client in three sentences, no jargon." *(For Quest 02: How did a dictionary let you find her best seller, and why did you count cups rather than transactions?)*

> 

**2. Associate audit:** "Where was your Associate wrong, misleading, or unhelpful this week? Paste the exchange." *(A draft that counted the wrong thing, a hallucinated method, an explanation that repeated itself without answering the question: Anything from your own chat.)*

> 

**3. Quiz me:** "Ask your Associate to set you three questions on this week's material. Paste the quiz and answer it *without* AI help." *(Suggested prompt: "Set me three short questions (easy, medium, hard). Cover Python dictionaries, loops over lists of dicts, and the difference between counting records and summing a field. Don't show answers until I've tried.")*

> 

### 📔 AI Use Record: Today's highlights

[SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) in your Codespace keeps the full transcript in `.specstory/history/`. Paste the **two** exchanges from today that a marker should see. Add one line of your own commentary to each. This habit becomes your AI Use Record in the assessments ([AI Guide, section 7](../../Guides/ai-guide.md)).

**Best exchange** *(the prompt that worked, and why it worked)*:

> **Me:** …
>
> **Associate:** …
>
> **Why it worked:** …

**Worst exchange** *(where the Associate was wrong, misleading, or useless, and how you caught it)*:

> **Me:** …
>
> **Associate:** …
>
> **How I caught it:** …

## 🧾 Signing off

Before you close the tab: **Run All** and check everything runs top to bottom. <!-- plain-language: everyday commit -->Commit and push your work. Write the [QuestLog](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-questlog) entry while it is fresh. Read the Debrief *after* you have attempted the tiers. Its failure <!-- plain-language: everyday Gallery -->gallery shows three mistakes your Associate could plausibly have made today. Check whether yours made any of them.